# Vaccination Coverage Analysis

## Project Overview
Analyze vaccination coverage by age group, region, and period to highlight gaps and possible equity issues.

## Learning Objectives
- Calculate vaccination coverage rates across demographic and geographic dimensions
- Identify under-vaccinated populations and equity gaps
- Analyze coverage trends over time
- Compare coverage across vaccine types

## Problem Statement
Public health agencies need to monitor vaccination coverage to identify under-served populations, plan outreach campaigns, and ensure equitable access to immunization.

## Why This Project Matters
Vaccination is one of the most cost-effective public health interventions. Coverage gaps leave populations vulnerable to outbreaks and exacerbate health inequities.

## Dataset Overview
Synthetic vaccination coverage dataset: quarterly coverage rates for 4 vaccines across 8 regions, 5 age groups over 4 years (2020-2023).

## Dataset Source and License Notes
- Synthetic data inspired by CDC vaccination coverage surveys
- No license restrictions

## Environment Setup

In [ ]:
import warnings
warnings.filterwarnings('ignore')
import matplotlib
matplotlib.use('Agg')

## Imports

In [ ]:
import numpy as np
import pandas as pd
import matplotlib.pyplot as plt
import seaborn as sns
sns.set_theme(style='whitegrid', palette='Set2')
np.random.seed(42)
print('Imports OK')

## Configuration / Constants

In [ ]:
import os
SAVE_DIR = os.path.dirname(os.path.abspath('__file__'))
print(f'Save dir: {SAVE_DIR}')

## Dataset Download or Loading

In [ ]:
np.random.seed(42)
regions = ['Northeast', 'Southeast', 'Midwest', 'Southwest', 'West Coast', 'Mountain', 'Gulf Coast', 'Mid-Atlantic']
age_groups = ['0-4', '5-17', '18-49', '50-64', '65+']
vaccines = ['Influenza', 'COVID-19', 'Childhood Series', 'Pneumococcal']
quarters = pd.date_range('2020-01-01', '2023-10-01', freq='QS')

rows = []
base_coverage = {
    ('Influenza', '0-4'): 70, ('Influenza', '5-17'): 55, ('Influenza', '18-49'): 35,
    ('Influenza', '50-64'): 48, ('Influenza', '65+'): 72,
    ('COVID-19', '0-4'): 15, ('COVID-19', '5-17'): 40, ('COVID-19', '18-49'): 55,
    ('COVID-19', '50-64'): 68, ('COVID-19', '65+'): 82,
    ('Childhood Series', '0-4'): 88, ('Childhood Series', '5-17'): 92,
    ('Childhood Series', '18-49'): 0, ('Childhood Series', '50-64'): 0, ('Childhood Series', '65+'): 0,
    ('Pneumococcal', '0-4'): 0, ('Pneumococcal', '5-17'): 0, ('Pneumococcal', '18-49'): 5,
    ('Pneumococcal', '50-64'): 25, ('Pneumococcal', '65+'): 68,
}
region_adj = {r: np.random.uniform(-8, 8) for r in regions}

for vaccine in vaccines:
    for age in age_groups:
        base = base_coverage.get((vaccine, age), 0)
        if base == 0:
            continue
        for region in regions:
            for q in quarters:
                trend = 0.5 * (q.year - 2020)  # slow improvement
                if vaccine == 'COVID-19' and q.year == 2020 and q.quarter <= 2:
                    coverage = 0
                else:
                    coverage = base + region_adj[region] + trend + np.random.normal(0, 3)
                coverage = np.clip(coverage, 0, 99).round(1)
                pop_segment = np.random.randint(50000, 500000)
                rows.append({
                    'Vaccine': vaccine, 'AgeGroup': age, 'Region': region,
                    'Quarter': q, 'CoveragePct': coverage,
                    'PopulationSegment': pop_segment,
                    'DosesAdministered': int(pop_segment * coverage / 100),
                })

df = pd.DataFrame(rows)
df['Year'] = df['Quarter'].dt.year

print(f'Dataset: {df.shape}')
print(f'Vaccines: {df["Vaccine"].nunique()}, Regions: {df["Region"].nunique()}')
df.head()

## Data Validation Checks

In [ ]:
print(f'Missing values: {df.isnull().sum().sum()}')
print(f'\nRecords by vaccine:\n{df["Vaccine"].value_counts()}')
print(f'\nCoverage stats:\n{df["CoveragePct"].describe().round(1)}')

## Overall Coverage by Vaccine and Age Group

In [ ]:
latest = df[df['Year'] == 2023]
pivot = latest.groupby(['Vaccine', 'AgeGroup'])['CoveragePct'].mean().unstack().round(1)
print('Avg Coverage % (2023) — Vaccine × Age Group:')
print(pivot)

fig, ax = plt.subplots(figsize=(10, 6))
sns.heatmap(pivot, annot=True, fmt='.0f', cmap='YlGn', ax=ax, vmin=0, vmax=100)
ax.set_title('Vaccination Coverage % (2023): Vaccine × Age Group')
plt.tight_layout()
plt.savefig(os.path.join(SAVE_DIR, 'coverage_heatmap.png'), dpi=100, bbox_inches='tight')
plt.show()

## Coverage Trends Over Time

In [ ]:
fig, axes = plt.subplots(2, 2, figsize=(14, 10))
for i, vaccine in enumerate(vaccines):
    ax = axes[i // 2, i % 2]
    sub = df[df['Vaccine'] == vaccine]
    for age in sub['AgeGroup'].unique():
        ag = sub[sub['AgeGroup'] == age].groupby('Quarter')['CoveragePct'].mean()
        ax.plot(ag.index, ag.values, label=age, marker='.', markersize=3)
    ax.set_title(f'{vaccine} Coverage Trend')
    ax.set_ylabel('Coverage %')
    ax.legend(fontsize=8)

plt.tight_layout()
plt.savefig(os.path.join(SAVE_DIR, 'coverage_trends.png'), dpi=100, bbox_inches='tight')
plt.show()

## Regional Coverage Gaps

In [ ]:
regional = latest.groupby(['Region', 'Vaccine'])['CoveragePct'].mean().unstack().round(1)
print('Regional Coverage (2023):')
print(regional)

fig, ax = plt.subplots(figsize=(12, 7))
sns.heatmap(regional, annot=True, fmt='.0f', cmap='RdYlGn', ax=ax, vmin=0, vmax=100)
ax.set_title('Regional Vaccination Coverage % (2023)')
plt.tight_layout()
plt.savefig(os.path.join(SAVE_DIR, 'regional_coverage.png'), dpi=100, bbox_inches='tight')
plt.show()

## Equity Gap Analysis

In [ ]:
# Compare highest vs lowest region for each vaccine
equity = df[df['Year'] == 2023].groupby(['Vaccine', 'Region'])['CoveragePct'].mean().unstack()
gap = equity.max(axis=1) - equity.min(axis=1)
print('Max Regional Coverage Gap by Vaccine:')
print(gap.round(1).sort_values(ascending=False))

# Age group gap
age_gap = df[df['Year'] == 2023].groupby(['Vaccine', 'AgeGroup'])['CoveragePct'].mean().unstack()
fig, ax = plt.subplots(figsize=(10, 5))
gap.sort_values().plot.barh(ax=ax, edgecolor='black', color='coral')
ax.set_title('Regional Coverage Gap (Max - Min) by Vaccine')
ax.set_xlabel('Coverage Gap (percentage points)')
plt.tight_layout()
plt.savefig(os.path.join(SAVE_DIR, 'equity_gap.png'), dpi=100, bbox_inches='tight')
plt.show()

## Childhood Vaccination Focus

In [ ]:
child = df[df['Vaccine'] == 'Childhood Series']
child_trend = child.groupby(['Year', 'Region'])['CoveragePct'].mean().unstack()
print('Childhood Series Coverage by Year × Region:')
print(child_trend.round(1))

below_90 = child[child['CoveragePct'] < 90].groupby('Region').size()
print(f'\nQuarters with <90% childhood coverage by region:')
print(below_90.sort_values(ascending=False))

## Interpretation and Business Insight
- **Influenza coverage** remains lowest among working-age adults (18-49) — workplace vaccination programs could help
- **COVID-19 coverage** shows strong age gradient — older adults are well-covered, but younger age groups lag significantly
- **Childhood series** maintains high coverage (>85%) but some regions are slipping below 90% — herd immunity threshold risk
- **Regional gaps** of 10-15 percentage points exist for most vaccines — geography-based outreach is needed
- **Pneumococcal** coverage among 65+ is good but 50-64 age group is under-vaccinated despite eligibility
- **Equity gaps** are widest for COVID-19 — indicating disparities in access or uptake

## Limitations
- Synthetic data with simplified coverage models
- No socioeconomic, racial, or insurance status stratification
- No urban/rural distinction within regions
- Coverage measured as point-in-time % rather than cumulative uptake
- No supply/access vs hesitancy distinction

## How to Improve This Project
- Add demographic stratification (race, income, insurance)
- Include urban/rural breakdown
- Add vaccine hesitancy survey data
- Track provider availability and distance-to-care metrics
- Model coverage targets needed for herd immunity thresholds

## Production Considerations
- Quarterly immunization coverage dashboards for public health agencies
- Automated alerts when coverage drops below herd immunity thresholds
- Geographic targeting for mobile vaccination clinics
- Integration with immunization registries for real-time tracking

## Common Mistakes
- Not normalizing by population when comparing coverage across regions
- Treating all vaccines the same way (different age targets, schedules)
- Ignoring the difference between doses administered and series completion
- Not distinguishing between access barriers and hesitancy

## Mini Challenge / Exercises
1. Which region has the most consistent coverage across all vaccines?
2. Calculate the number of additional doses needed to bring all regions to 80% COVID-19 coverage for 18-49.
3. What is the year-over-year improvement rate for Childhood Series nationally?
4. If coverage dropped 5% in all regions, which vaccine would be closest to losing herd immunity protection?

## Final Summary / Key Takeaways
- Vaccination coverage analysis reveals who is protected and who is not
- Age group and regional stratification are essential for identifying gaps
- Equity gaps between regions highlight access and outreach disparities
- Childhood vaccination coverage must be closely monitored to maintain herd immunity
- Trend analysis over time shows whether public health interventions are working